In [1]:
import importlib
import sys
import pickle

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')

from model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM

In [2]:
#load model
file_path_model = '../../../training_variational_dropout_v2/Repair/Repair_full_grad_norm_pro_conf_check_v2.pkl'
model = DropoutUncertaintyEncoderDecoderLSTM.load(file_path_model, dropout=0.1)

# Load the dataset
# file_path_data_set = '../../../../../encoded_data_conformance_v2/Repair/repair_all_3_val.pkl'
# repair_test_dataset = torch.load(file_path_data_set, weights_only=False)

file_path_data_set = '../../../../../encoded_data_conformance_v2/Repair/repair_all_3_test.pkl'
repair_test_dataset = torch.load(file_path_data_set, weights_only=False)

Dynamic data set categories:  ([('concept:name', 9, {'ACKNOWLEDGEMENT': 1, 'CREATE_INVOICE': 2, 'DISASSEMBLY': 3, 'EOS': 4, 'QUALITY_CONTROL': 5, 'RECEPTION': 6, 'REPAIR': 7, 'SHIPPING': 8}), ('org:resource', 52, {'0': 1, '1': 2, '10': 3, '11': 4, '12': 5, '13': 6, '14': 7, '15': 8, '16': 9, '17': 10, '18': 11, '19': 12, '2': 13, '20': 14, '21': 15, '22': 16, '23': 17, '24': 18, '25': 19, '26': 20, '27': 21, '28': 22, '29': 23, '3': 24, '30': 25, '31': 26, '32': 27, '33': 28, '34': 29, '35': 30, '36': 31, '37': 32, '38': 33, '39': 34, '4': 35, '40': 36, '41': 37, '42': 38, '43': 39, '44': 40, '45': 41, '46': 42, '47': 43, '48': 44, '49': 45, '5': 46, '6': 47, '7': 48, '8': 49, '9': 50, 'EOS': 51})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {}), ('day_in_week', 1, {}), ('seconds_in_day', 1, {})])
Data set static categories:  ([], [])
Encoder dynamic input features:  [['concept:name', 'org:resource'], ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_

In [3]:
import evaluation_v2.probabilistic_evaluation
importlib.reload(evaluation_v2.probabilistic_evaluation)
from evaluation_v2.probabilistic_evaluation import ProbabilisticEvaluation

new_eval = ProbabilisticEvaluation(model=model, 
                                   dataset=repair_test_dataset,
                                   concept_name='concept:name',
                                   num_processes=32,
                                   growing_num_values = [],
                                   # growing_num_values = ['case_elapsed_time'],
                                   # number of samples
                                   samples_per_case = 100,
                                   sample_argmax = False,
                                   use_variance_cat = True,
                                   use_variance_num = True,
                                   decoder_cat=['concept:name'],
                                   decoder_num=['event_elapsed_time']
                                   )

In [4]:
def save_chunk(results, i):
    chunk_number = (i + 1)
    filename = os.path.join(output_dir, f'results_part_{chunk_number:03d}.pkl')
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} results to {filename}")

# output_dir = '../../../../../../../data/repair_shop/proact_conf_check_v2/eval_validation_set'
output_dir = '../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set'

save_every = 50

results = {}
#for i, (case_name, prefix_len, prefix, predicted_suffixes, suffix, mean_prediction) in enumerate(new_eval.evaluate(random_order=True)):
for i, (case_name, prefix_len, prefix, predicted_suffixes, suffix, mean_prediction) in enumerate(new_eval.evaluate_multi_processing(random_order=True)):
    # print(case_name, prefix_len)
    assert((case_name, prefix_len) not in results)
    results[(case_name, prefix_len)] = (prefix, suffix, mean_prediction, predicted_suffixes)
    # print(prefix_len, len(suffix))
    if (i + 1) % save_every == 0:
        save_chunk(results, i)
        results = {}

if len(results):
    save_chunk(results, i)

  0%|          | 0/1496 [00:00<?, ?it/s]

Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_050.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_100.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_150.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_200.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_250.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_300.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_350.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test_set/results_part_400.pkl
Saved 50 results to ../../../../../../../data/repair_shop/proact_conf_check_v2/eval_test